### store_inventory_snapshots table ingestion into bronze layer from data source

In [0]:
# Defining schema for snapshot table

from pyspark.sql.types import *
from pyspark.sql import functions as F

snapshot_schema = StructType([
    StructField("store_id", StringType(), False),
    StructField("product_id", StringType(), False),
    StructField("current_quantity", IntegerType(), True),
    StructField("last_restocked_date", DateType(), True),
    StructField("shelf_location", StringType(), True),
    StructField("expiry_date", DateType(), True),
    StructField("temperature_reading", StringType(), True)   # nested JSON stored as string
])

In [0]:
# Read JSONL as Streaming Source

df_snapshots = (
    spark.readStream
        .format("cloudFiles")
        .option("cloudFiles.format", "json")
        .schema(snapshot_schema)
        .option("cloudFiles.schemaLocation", "/Volumes/mini_project_team_a3t7/default/mini_project/schema/store_inventory_snapshots/")
        .option("maxFilesPerTrigger", 10)
        .load("/Volumes/mini_project_team_a3t7/default/mini_project/raw/store_inventory_snapshots/")
)

In [0]:
# Adding Metadata column

from pyspark.sql import functions as F

df_bronze = (
    df_snapshots
        .withColumn("_ingestion_timestamp", F.current_timestamp())
        .withColumn("_batch_date", F.current_date())
        .withColumn("_source", F.lit("store_inventory_snapshots"))
        .withColumn("_source_file", F.col("_metadata.file_path"))
)



In [0]:
# Writing to Bronze Table

query = (
    df_bronze.writeStream
        .format("delta")
        .outputMode("append")
        .option(
            "checkpointLocation",
            "/Volumes/mini_project_team_a3t7/default/mini_project/checkpoint/bronze/store_inventory_stream"
        )
        .partitionBy("_batch_date")
        .trigger(availableNow=True)
        # .trigger(processingTime="15 minutes")
        .start("/Volumes/mini_project_team_a3t7/default/mini_project/bronze/store_inventory_stream")
)
